In [47]:
## import dependencies
import os
import re
import sys
import json
import h5py
import numpy as np
import matplotlib.pyplot as plt
import imageio_ffmpeg

from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from IPython.display import HTML

plt.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

## Add the path to the DiscEvolution directory
sys.path.append('/Users/ben/Downloads/Planet Formation/Code/DiscEvolution Core Accretion')

## import DiscEvolution modules
from DiscEvolution.constants import *
from DiscEvolution.grid import Grid
from DiscEvolution.star import SimpleStar
from DiscEvolution.eos import IrradiatedEOS, LocallyIsothermalEOS, SimpleDiscEOS
from DiscEvolution.disc import *
from DiscEvolution.viscous_evolution import ViscousEvolution, ViscousEvolutionFV, LBP_Solution, HybridWindModel, TaboneSolution
from DiscEvolution.disc import AccretionDisc
from DiscEvolution.dust import *
from DiscEvolution.dust import PlanetesimalFormation
from DiscEvolution.planet_formation import *
from DiscEvolution.diffusion import TracerDiffusion
from DiscEvolution.opacity import Tazzari2016
from DiscEvolution.chemistry import *

In [48]:
## Method to read data from an HDF5 dataset or group

def read_hdf5_array(obj):
    """
    Read data from either an HDF5 dataset or an HDF5 group.
    """

    ## Normal case (the data are stored as one HDF5 dataset)
    if isinstance(obj, h5py.Dataset):
        return np.array(obj)

    ## Ragged case (the data are stored as a group)
    elif isinstance(obj, h5py.Group):
        values = []

        for key in sorted(obj.keys(), key = int):
            values.append(np.array(obj[key]))

        return values

    ## Raise an error if an unexpected HDF5 object is encountered
    else:
        raise TypeError(f"Unexpected HDF5 object type: {type(obj)}")

## Method to load data from an HDF5 file, together with the config used to produce it

def load_data(data_dir, config_dir):
    """
    Load a disc evolution output file (HDF5 only).

    Parameters:
        data_dir: Path to the HDF5 output file.
        config_dir: Path to the JSON config file used for the run.

    Returns:
        data: Dictionary containing the simulation output.
        config: Dictionary containing the simulation config file.
    """

    data = {}

    ## Load the configuration file

    with open(config_dir, "r") as f:
        config = json.load(f)

    ## Open the HDF5 file in read-only mode
    with h5py.File(data_dir, "r") as f:

        ## -----------------------
        ## General simulation data
        ## -----------------------

        general_keys = [
            "R",
            "t",
            "disk_Mdot_star",
            "disk_Mass",
            "Tc",
            "Sigc"]

        for key in general_keys:
            ## Only load the quantity if it exists in the file
            if key in f:
                data[key] = np.array(f[key])

        ## ---------------
        ## Alpha viscosity
        ## ---------------

        if "alpha_SS" in f.attrs:
            data["alpha_SS"] = float(f.attrs["alpha_SS"])

        elif "alpha_SS" in f:
            data["alpha_SS"] = float(f["alpha_SS"][()])

        if config["winds"]["on"]:
            data["psi"] = config["winds"]["psi_DW"]

        ## -------------
        ## Disk profiles
        ## -------------

        disk_keys = [
            "T",
            "time_snap",
            "Sigma_G",
            "Sigma_dust",
            "Sigma_pebbles",
            "Sigma_planetesimals",
            "Vdrift_grains",
            "Vdrift_pebbles",
            "St_grains",
            "St_pebbles",
            "St_planetesimals",
            "e_planetesimals",
            "i_planetesimals",
            "de2_dt",
            "di2_dt",
            "de2_dt_drag",
            "di2_dt_drag",
            "de2_dt_VS_embryo",
            "di2_dt_VS_embryo",
            "de2_dt_VS_pltsml",
            "di2_dt_VS_pltsml",
            "de2_dt_DF",
            "di2_dt_DF"]

        for key in disk_keys:
            ## Only load the quantity if it exists in the file
            if key in f:
                data[key] = read_hdf5_array(f[key])

        ## -----------
        ## Planet data
        ## -----------

        if config["planets"]["include_planets"]:
            planet_keys = [
                "Mcs",
                "Mes",
                "Rp",
                "disk_Mdot_p",
                "Mdot_planetesimal",
                "Mdot_pebble_core",
                "Mdot_pebble_env",
                "Mdot_migration",
                "Mdot_gas",
                "M_iso_planetesimal",
                "M_iso_pebble",
                "X_cores",
                "X_envs",
                "t_form"]

            for key in planet_keys:
                if key in f:
                    data[key] = read_hdf5_array(f[key])

    return data, config

In [49]:
## Load data and config file

data_dir = "/Users/ben/Downloads/Planet Formation/DiscEvolution Simulations/Output/20260812_0428_psi10_Mdot1.0e-08_M0.1_Rd50.h5"
config_dir = "/Users/ben/Downloads/Planet Formation/DiscEvolution Simulations/Config/20260810_full_accretion.json"

data, config = load_data(data_dir, config_dir)

In [50]:
## Figure output

config_name = os.path.splitext(os.path.basename(config_dir))[0]
figure_dir = f"/Users/ben/Downloads/Planet Formation/DiscEvolution Movies/{config_name}/"

if not(os.path.exists(figure_dir)):
        os.makedirs(figure_dir)

save_figures = True

In [51]:
## Extract data for plotting

## Disc profiles
R = data["R"]
alpha = data["alpha_SS"]

## Surface density profiles
Sigma_G = data["Sigma_G"]
Sigma_grains = data["Sigma_dust"]
Sigma_pebble = data["Sigma_pebbles"]

if config["planetesimal"]["active"]:
    Sigma_planetesimal = data["Sigma_planetesimals"]

## Temperature profiles
T = data["T"]

## Eccentricity and inclination profiles
if config["planetesimal"]["active"]:
    e_planetesimal = data["e_planetesimals"]
    i_planetesimal = data["i_planetesimals"]

## Eccentricity and inclination derivatives
if config["planetesimal"]["active"]:
    if config["planetesimal"]["drag"]:
        de2_dt_drag = data["de2_dt_drag"]
        di2_dt_drag = data["di2_dt_drag"]

    if config["planetesimal"]["VS_embryo"]:
        de2_dt_VS_embryo = data["de2_dt_VS_embryo"]
        di2_dt_VS_embryo = data["di2_dt_VS_embryo"]

    if config["planetesimal"]["VS_pltsml"]:
        de2_dt_VS_pltsml = data["de2_dt_VS_pltsml"]
        di2_dt_VS_pltsml = data["di2_dt_VS_pltsml"]

    if config["planetesimal"]["DF"]:
        de2_dt_DF = data["de2_dt_DF"]
        di2_dt_DF = data["di2_dt_DF"]

    if config["planetesimal"]["drag"] or config["planetesimal"]["VS_embryo"] or config["planetesimal"]["VS_pltsml"] or config["planetesimal"]["DF"]:
        de2_dt = data["de2_dt"]
        di2_dt = data["di2_dt"]

## Accretion rates
if config["planets"]["include_planets"]:
    if config["planets"]["planetesimal_accretion_insitu"]:
        Mdot_planetesimal = data["Mdot_planetesimal"]

    if config["planets"]["planetesimal_accretion_migrate"]:
        Mdot_migration = data["Mdot_migration"]

    if config["planets"]["pebble_accretion"]:
        Mdot_pebble_core = data["Mdot_pebble_core"]
        Mdot_pebble_env = data["Mdot_pebble_env"]

    if config["planets"]["gas_accretion"]:
        Mdot_gas = data["Mdot_gas"]

## Planet properties
if config["planets"]["include_planets"]:
    R_imp = config["planets"]["Rp"]
    Rp = data["Rp"]
    Mcs = data["Mcs"]
    Mes = data["Mes"]
    Mp = Mcs + Mes
    t_form = data["t_form"]

    if config["planets"]["planetesimal_accretion_insitu"]:
        M_iso_planetesimal = data["M_iso_planetesimal"]

    if config["planets"]["pebble_accretion"]:
        M_iso_pebble = data["M_iso_pebble"]

## Time data
t = data["t"]
t_interval = config["simulation"]["t_interval"]

In [52]:
## Matplotlib colors

n_times = len(t_interval)

color_blues = plt.cm.Blues(np.linspace(1, 0.4, n_times))
color_greys = plt.cm.Greys(np.linspace(1, 0.4, n_times))
color_greens = plt.cm.Greens(np.linspace(1, 0.4, n_times))
color_reds = plt.cm.Reds(np.linspace(1, 0.4, n_times))

In [53]:
## Animate surface density profiles (one line per timestep)

n_times = len(t_interval)

fig, ax = plt.subplots(2, 2, figsize = (8 * 2, 6 * 2))

lines = {}
lines['gas'], = ax[0, 0].plot([], [], color = 'tab:blue')
lines['grains'], = ax[0, 1].plot([], [], color = 'tab:grey')
lines['pebbles'], = ax[1, 0].plot([], [], color = 'tab:green')

if config["planetesimal"]["active"]:
    lines['planetesimals'], = ax[1, 1].plot([], [], color = 'tab:red')

for i in range(2):
    for j in range(2):
        ax[i, j].set_xscale('log')
        ax[i, j].set_yscale('log')
        ax[i, j].set_xlim(R.min(), R.max())
        ax[i, j].set_ylim(1e-18, 1e6)
        ax[i, j].set_xlabel('Radius [AU]')
        ax[i, j].set_ylabel(r'Surface Density [$\rm{g/cm^2}$]')

ax[0, 0].set_title('Gas')
ax[0, 1].set_title('Grains')
ax[1, 0].set_title('Pebbles')
ax[1, 1].set_title('Planetesimals')

time_title = fig.suptitle('', fontsize = 16)

plt.tight_layout()

def update(i):
    lines['gas'].set_data(R, Sigma_G[i])
    lines['grains'].set_data(R, Sigma_grains[i])
    lines['pebbles'].set_data(R, Sigma_pebble[i])

    if config["planetesimal"]["active"]:
        lines['planetesimals'].set_data(R, Sigma_planetesimal[i])

    time_title.set_text(f't = {t_interval[i]} Myr')

    return list(lines.values()) + [time_title]

ani = FuncAnimation(fig, update, frames = n_times, interval = 1000, blit = False)

if save_figures:
    gif_path = os.path.join(figure_dir, "surface_density_profiles.gif")
    ani.save(gif_path, writer = PillowWriter(fps = 5))

    mp4_path = os.path.join(figure_dir, "surface_density_profiles.mp4")
    ani.save(mp4_path, writer = FFMpegWriter(fps = 5))

plt.close(fig)

HTML(ani.to_jshtml())

In [54]:
## Pressure profile helper functions

def Omega_K(R):
    """Calculates the Keplerian angular velocity of the disk in rad/s."""
    M_star = 1.0 * Msun
    G = 6.67430e-8
    return np.sqrt(G * M_star / (R * AU) ** 3)

def rho_mid(R, Sigma_G, T):
    """Calculates the midplane gas density of the disk in g/cm^3."""
    return Sigma_G / (np.sqrt(2 * np.pi) * H(R, T) * AU)

def c_s(T):
    """Calculates the sound speed of the disk in cm/s."""
    mu = 2.33
    return np.sqrt(k_B * T / (mu * m_H))

def H(R, T):
    """Calculates the scale height of the disk in AU."""
    return c_s(T) / Omega_K(R) / AU

def P(R, Sigma_G, T):
    """Calculates the pressure of the disk in g/cm/s^2 assuming an ideal gas."""
    return rho_mid(R, Sigma_G, T) * c_s(T) ** 2

def dP_dR(R, Sigma_G, T):
    """Calculates the pressure gradient of the disk."""
    return np.gradient(P(R, Sigma_G, T), R * AU)

def dlnP_dlnR(R, Sigma_G, T):
    """Calculates dlnP/dlnR of the disk."""
    return (R * AU) / P(R, Sigma_G, T) * dP_dR(R, Sigma_G, T)

In [55]:
## Animate temperature and pressure profiles (one line per timestep)

n_times = len(t_interval)

P_all = np.array([P(R, Sigma_G[i], T[i]) for i in range(n_times)])

T_min, T_max = np.nanmin(T), np.nanmax(T)
P_min, P_max = np.nanmin(P_all), np.nanmax(P_all)

fig, ax = plt.subplots(1, 2, figsize = (8 * 2, 6))

lines = {}
lines['T'], = ax[0].plot([], [], color = 'tab:blue')
lines['P'], = ax[1].plot([], [], color = 'tab:red')

ax[0].axhline(150, color = '#1f77b4', linestyle = '--', label = r'H$_2$O (150 K)', alpha = 0.7)
ax[0].axhline(70, color = '#ff7f0e', linestyle = '--', label = r'CO$_2$ (70 K)', alpha = 0.7)
ax[0].axhline(20, color = '#d62728', linestyle = '--', label = r'CO (20 K)', alpha = 0.7)

ax[0].set_xlim(R.min(), R.max())
ax[0].set_ylim(T_min * 0.8, T_max * 1.2)
ax[1].set_xlim(R.min(), R.max())
ax[1].set_ylim(P_min * 0.8, P_max * 1.2)

for i in range(2):
    ax[i].set_xscale('log')
    ax[i].set_yscale('log')
    ax[i].set_xlabel('Radius [AU]')

ax[0].set_ylabel('Temperature [K]')
ax[1].set_ylabel(r'Pressure [$\rm{g/cm/s^2}$]')
ax[0].set_title('Temperature Profile')
ax[1].set_title('Pressure Profile')
ax[0].legend(loc = 'upper right', fontsize = 8)

time_title = fig.suptitle('', fontsize = 16)

plt.tight_layout()

def update(i):
    lines['T'].set_data(R, T[i])
    lines['P'].set_data(R, P_all[i])

    time_title.set_text(f't = {t_interval[i]} Myr')

    return list(lines.values()) + [time_title]

ani = FuncAnimation(fig, update, frames = n_times, interval = 1000, blit = False)

if save_figures:
    gif_path = os.path.join(figure_dir, "temperature_pressure_profiles.gif")
    ani.save(gif_path, writer = PillowWriter(fps = 5))

    mp4_path = os.path.join(figure_dir, "temperature_pressure_profiles.mp4")
    ani.save(mp4_path, writer = FFMpegWriter(fps = 5))

plt.close(fig)

HTML(ani.to_jshtml())

In [56]:
## Animate eccentricity and inclination profiles (one line per timestep)

n_times = len(t_interval)

fig, ax = plt.subplots(1, 2, figsize = (8 * 2, 6))

lines = {}

if config["planetesimal"]["active"]:
    lines['e'], = ax[0].plot([], [], color = 'tab:blue')
    lines['i'], = ax[1].plot([], [], color = 'tab:red')

    e_min, e_max = np.nanmin(e_planetesimal), np.nanmax(e_planetesimal)
    i_min, i_max = np.nanmin(i_planetesimal), np.nanmax(i_planetesimal)

    ax[0].set_ylim(e_min * 0.8, e_max * 1.2)
    ax[1].set_ylim(i_min * 0.8, i_max * 1.2)

for i in range(2):
    ax[i].set_xscale('log')
    ax[i].set_yscale('log')
    ax[i].set_xlim(R.min(), R.max())
    ax[i].set_xlabel('Radius [AU]')

ax[0].set_title('Eccentricity')
ax[1].set_title('Inclination')

time_title = fig.suptitle('', fontsize = 16)

plt.tight_layout()

def update(i):
    if config["planetesimal"]["active"]:
        lines['e'].set_data(R, e_planetesimal[i])
        lines['i'].set_data(R, i_planetesimal[i])

    time_title.set_text(f't = {t_interval[i]} Myr')

    return list(lines.values()) + [time_title]

ani = FuncAnimation(fig, update, frames = n_times, interval = 1000, blit = False)

if save_figures:
    gif_path = os.path.join(figure_dir, "eccentricity_inclination_profiles.gif")
    ani.save(gif_path, writer = PillowWriter(fps = 5))

    mp4_path = os.path.join(figure_dir, "eccentricity_inclination_profiles.mp4")
    ani.save(mp4_path, writer = FFMpegWriter(fps = 5))

plt.close(fig)

HTML(ani.to_jshtml())

In [57]:
## Animate eccentricity and inclination derivatives (one line per timestep)

n_times = len(t_interval)

de2_abs = np.abs(de2_dt)
di2_abs = np.abs(di2_dt)

de2_min, de2_max = np.nanmin(de2_abs[de2_abs > 0]), np.nanmax(de2_abs[de2_abs > 0])
di2_min, di2_max = np.nanmin(di2_abs[di2_abs > 0]), np.nanmax(di2_abs[di2_abs > 0])

fig, ax = plt.subplots(1, 2, figsize = (8 * 2, 6))

lines = {}
lines['de_pos'], = ax[0].plot([], [], color = 'tab:blue', linestyle = '-')
lines['de_neg'], = ax[0].plot([], [], color = 'tab:blue', linestyle = '--')
lines['di_pos'], = ax[1].plot([], [], color = 'tab:red', linestyle = '-')
lines['di_neg'], = ax[1].plot([], [], color = 'tab:red', linestyle = '--')

ax[0].set_xlim(R.min(), R.max())
ax[0].set_ylim(de2_min * 0.8, de2_max * 1.2)
ax[1].set_xlim(R.min(), R.max())
ax[1].set_ylim(di2_min * 0.8, di2_max * 1.2)

for i in range(2):
    ax[i].set_xscale('log')
    ax[i].set_yscale('log')
    ax[i].set_xlabel('Radius [AU]')
    ax[i].set_ylabel('[1/yr]')

ax[0].set_title('Eccentricity Derivatives')
ax[1].set_title('Inclination Derivatives')

time_title = fig.suptitle('', fontsize = 16)

plt.tight_layout()

def update(i):
    lines['de_pos'].set_data(R, np.where(de2_dt[i] > 0, de2_dt[i], np.nan))
    lines['de_neg'].set_data(R, np.where(de2_dt[i] < 0, -1 * de2_dt[i], np.nan))
    lines['di_pos'].set_data(R, np.where(di2_dt[i] > 0, di2_dt[i], np.nan))
    lines['di_neg'].set_data(R, np.where(di2_dt[i] < 0, -1 * di2_dt[i], np.nan))

    time_title.set_text(f't = {t_interval[i]} Myr')

    return list(lines.values()) + [time_title]

ani = FuncAnimation(fig, update, frames = n_times, interval = 1000, blit = False)

if save_figures:
    gif_path = os.path.join(figure_dir, "de2dt_di2dt_profiles.gif")
    ani.save(gif_path, writer = PillowWriter(fps = 5))

    mp4_path = os.path.join(figure_dir, "de2dt_di2dt_profiles.mp4")
    ani.save(mp4_path, writer = FFMpegWriter(fps = 5))

plt.close(fig)

HTML(ani.to_jshtml())

In [58]:
## Animate eccentricity and inclination derivative components (one line per timestep)

n_times = len(t_interval)

has_derivatives = config["planetesimal"]["drag"] or config["planetesimal"]["VS_embryo"] or config["planetesimal"]["VS_pltsml"] or config["planetesimal"]["DF"]

fig, ax = plt.subplots(1, 2, figsize = (8 * 2, 6))

lines = {}

if has_derivatives:
    lines['de_total_pos'], = ax[0].plot([], [], color = 'black', linestyle = '-', label = r'$\dot e^2(t)$')
    lines['de_total_neg'], = ax[0].plot([], [], color = 'black', linestyle = '--')
    lines['di_total_pos'], = ax[1].plot([], [], color = 'black', linestyle = '-', label = r'$\dot i^2(t)$')
    lines['di_total_neg'], = ax[1].plot([], [], color = 'black', linestyle = '--')

if config["planetesimal"]["drag"]:
    lines['de_drag_pos'], = ax[0].plot([], [], color = 'blue', linestyle = '-', label = 'Drag')
    lines['de_drag_neg'], = ax[0].plot([], [], color = 'blue', linestyle = '--')
    lines['di_drag_pos'], = ax[1].plot([], [], color = 'blue', linestyle = '-', label = 'Drag')
    lines['di_drag_neg'], = ax[1].plot([], [], color = 'blue', linestyle = '--')

if config["planetesimal"]["VS_embryo"]:
    lines['de_VS_embryo_pos'], = ax[0].plot([], [], color = 'red', linestyle = '-', label = 'Embryo Stirring')
    lines['de_VS_embryo_neg'], = ax[0].plot([], [], color = 'red', linestyle = '--')
    lines['di_VS_embryo_pos'], = ax[1].plot([], [], color = 'red', linestyle = '-', label = 'Embryo Stirring')
    lines['di_VS_embryo_neg'], = ax[1].plot([], [], color = 'red', linestyle = '--')

if config["planetesimal"]["VS_pltsml"]:
    lines['de_VS_pltsml_pos'], = ax[0].plot([], [], color = 'orange', linestyle = '-', label = 'Planetesimal Stirring')
    lines['de_VS_pltsml_neg'], = ax[0].plot([], [], color = 'orange', linestyle = '--')
    lines['di_VS_pltsml_pos'], = ax[1].plot([], [], color = 'orange', linestyle = '-', label = 'Planetesimal Stirring')
    lines['di_VS_pltsml_neg'], = ax[1].plot([], [], color = 'orange', linestyle = '--')

if config["planetesimal"]["DF"]:
    lines['de_DF_pos'], = ax[0].plot([], [], color = 'green', linestyle = '-', label = 'Density Fluctuations')
    lines['de_DF_neg'], = ax[0].plot([], [], color = 'green', linestyle = '--')
    lines['di_DF_pos'], = ax[1].plot([], [], color = 'green', linestyle = '-', label = 'Density Fluctuations')
    lines['di_DF_neg'], = ax[1].plot([], [], color = 'green', linestyle = '--')

de2_arrays = []
di2_arrays = []

if has_derivatives:
    de2_arrays.append(de2_dt)
    di2_arrays.append(di2_dt)

if config["planetesimal"]["drag"]:
    de2_arrays.append(de2_dt_drag)
    di2_arrays.append(di2_dt_drag)

if config["planetesimal"]["VS_embryo"]:
    de2_arrays.append(de2_dt_VS_embryo)
    di2_arrays.append(di2_dt_VS_embryo)

if config["planetesimal"]["VS_pltsml"]:
    de2_arrays.append(de2_dt_VS_pltsml)
    di2_arrays.append(di2_dt_VS_pltsml)

if config["planetesimal"]["DF"]:
    de2_arrays.append(de2_dt_DF)
    di2_arrays.append(di2_dt_DF)

de2_abs = np.abs(np.concatenate([arr.ravel() for arr in de2_arrays]))
di2_abs = np.abs(np.concatenate([arr.ravel() for arr in di2_arrays]))

de2_min, de2_max = np.nanmin(de2_abs[de2_abs > 0]), np.nanmax(de2_abs[de2_abs > 0])
di2_min, di2_max = np.nanmin(di2_abs[di2_abs > 0]), np.nanmax(di2_abs[di2_abs > 0])

ax[0].set_xlim(R.min(), R.max())
ax[0].set_ylim(de2_min * 0.8, de2_max * 1.2)
ax[1].set_xlim(R.min(), R.max())
ax[1].set_ylim(di2_min * 0.8, di2_max * 1.2)

for i in range(2):
    ax[i].set_xscale('log')
    ax[i].set_yscale('log')
    ax[i].set_xlabel('Radius [AU]')
    ax[i].set_ylabel('[1/yr]')
    ax[i].legend(loc = 'lower left', fontsize = 8)

ax[0].set_title('Eccentricity Derivatives')
ax[1].set_title('Inclination Derivatives')

time_title = fig.suptitle('', fontsize = 16)

plt.tight_layout()

def update(i):
    if has_derivatives:
        lines['de_total_pos'].set_data(R, np.where(de2_dt[i] > 0, de2_dt[i], np.nan))
        lines['de_total_neg'].set_data(R, np.where(de2_dt[i] < 0, -1 * de2_dt[i], np.nan))
        lines['di_total_pos'].set_data(R, np.where(di2_dt[i] > 0, di2_dt[i], np.nan))
        lines['di_total_neg'].set_data(R, np.where(di2_dt[i] < 0, -1 * di2_dt[i], np.nan))

    if config["planetesimal"]["drag"]:
        lines['de_drag_pos'].set_data(R, np.where(de2_dt_drag[i] > 0, de2_dt_drag[i], np.nan))
        lines['de_drag_neg'].set_data(R, np.where(de2_dt_drag[i] < 0, -1 * de2_dt_drag[i], np.nan))
        lines['di_drag_pos'].set_data(R, np.where(di2_dt_drag[i] > 0, di2_dt_drag[i], np.nan))
        lines['di_drag_neg'].set_data(R, np.where(di2_dt_drag[i] < 0, -1 * di2_dt_drag[i], np.nan))

    if config["planetesimal"]["VS_embryo"]:
        lines['de_VS_embryo_pos'].set_data(R, np.where(de2_dt_VS_embryo[i] > 0, de2_dt_VS_embryo[i], np.nan))
        lines['de_VS_embryo_neg'].set_data(R, np.where(de2_dt_VS_embryo[i] < 0, -1 * de2_dt_VS_embryo[i], np.nan))
        lines['di_VS_embryo_pos'].set_data(R, np.where(di2_dt_VS_embryo[i] > 0, di2_dt_VS_embryo[i], np.nan))
        lines['di_VS_embryo_neg'].set_data(R, np.where(di2_dt_VS_embryo[i] < 0, -1 * di2_dt_VS_embryo[i], np.nan))

    if config["planetesimal"]["VS_pltsml"]:
        lines['de_VS_pltsml_pos'].set_data(R, np.where(de2_dt_VS_pltsml[i] > 0, de2_dt_VS_pltsml[i], np.nan))
        lines['de_VS_pltsml_neg'].set_data(R, np.where(de2_dt_VS_pltsml[i] < 0, -1 * de2_dt_VS_pltsml[i], np.nan))
        lines['di_VS_pltsml_pos'].set_data(R, np.where(di2_dt_VS_pltsml[i] > 0, di2_dt_VS_pltsml[i], np.nan))
        lines['di_VS_pltsml_neg'].set_data(R, np.where(di2_dt_VS_pltsml[i] < 0, -1 * di2_dt_VS_pltsml[i], np.nan))

    if config["planetesimal"]["DF"]:
        lines['de_DF_pos'].set_data(R, np.where(de2_dt_DF[i] > 0, de2_dt_DF[i], np.nan))
        lines['de_DF_neg'].set_data(R, np.where(de2_dt_DF[i] < 0, -1 * de2_dt_DF[i], np.nan))
        lines['di_DF_pos'].set_data(R, np.where(di2_dt_DF[i] > 0, di2_dt_DF[i], np.nan))
        lines['di_DF_neg'].set_data(R, np.where(di2_dt_DF[i] < 0, -1 * di2_dt_DF[i], np.nan))

    time_title.set_text(f't = {t_interval[i]} Myr')

    return list(lines.values()) + [time_title]

ani = FuncAnimation(fig, update, frames = n_times, interval = 1000, blit = False)

if save_figures:
    gif_path = os.path.join(figure_dir, "de2dt_di2dt_components_profiles.gif")
    ani.save(gif_path, writer = PillowWriter(fps = 5))

    mp4_path = os.path.join(figure_dir, "de2dt_di2dt_components_profiles.mp4")
    ani.save(mp4_path, writer = FFMpegWriter(fps = 5))

plt.close(fig)

HTML(ani.to_jshtml())

In [59]:
## Animate planet growth tracks, colored by envelope-to-core mass fraction (one frame per timestep)

n_planets = len(Rp)
n_times = len(t_interval)

cmap = plt.get_cmap('plasma')
norm = Normalize(vmin = 0, vmax = 1)

x_min, x_max = 1e-1, 1e2 / 2

y_all = np.concatenate([Mcs[i] + Mes[i] for i in range(n_planets)])
y_min, y_max = np.nanmin(y_all[y_all > 0]), np.nanmax(y_all)

fig, ax = plt.subplots(figsize = (8, 6))

sm = plt.cm.ScalarMappable(cmap = cmap, norm = norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax = ax)
cbar.set_label('Envelope Mass Fraction')

def update(j):
    ax.clear()

    t_now = t_interval[j] * 1e6

    for i in range(n_planets):
        mask = t <= t_now

        x = Rp[i][mask]
        y = (Mcs[i] + Mes[i])[mask]
        frac = (Mes[i] / (Mcs[i] + Mes[i]))[mask]

        if len(x) > 1:
            points = np.array([x, y]).T.reshape(-1, 1, 2)
            segments = np.concatenate([points[:-1], points[1:]], axis = 1)

            lc = LineCollection(segments, cmap = cmap, norm = norm)
            lc.set_array(frac[:-1])
            lc.set_linewidth(2)
            ax.add_collection(lc)

        if len(x) > 0:
            ax.scatter(x[-1], y[-1], color = 'black', s = 30, zorder = 3)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min * 0.8, y_max * 1.2)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Radial Distance [AU]')
    ax.set_ylabel(r'Total Mass [M$_\oplus$]')
    ax.set_title(f'Planet Growth Tracks: t = {t_interval[j]} Myr')

update(0)
plt.tight_layout()

ani = FuncAnimation(fig, update, frames = n_times, interval = 1000, blit = False)

if save_figures:
    gif_path = os.path.join(figure_dir, "planet_growth_tracks.gif")
    ani.save(gif_path, writer = PillowWriter(fps = 5))

    mp4_path = os.path.join(figure_dir, "planet_growth_tracks.mp4")
    ani.save(mp4_path, writer = FFMpegWriter(fps = 5))

plt.close(fig)

HTML(ani.to_jshtml())

In [60]:
## Streaming instability birth mass

def M_birth(R, Sigma_G, T):
    """Calculates the streaming instability brith mass in Earth masses (Johnston et al 2026)."""
    M_star = 1.0
    return 6e-2 * (gamma(R, Sigma_G, T) * np.pi) ** 1.5 * (H(R, T) / R / 0.05) ** 3 * (M_star / 2.4)

def gamma(R, Sigma_G, T):
    """Calculates the self-gravity parameter (Johnston et al 2026)."""
    G = 6.67430e-8
    return 4 * np.pi * G * rho_mid(R, Sigma_G, T) / (Omega_K(R) ** 2)

## Pebble accretion onset mass

def M_on(R, Sigma_G, T):
    """Calculates the onset mass for pebble accretion in Earth masses (Lorek et al 2022)."""
    M_star = 1.0  # in units of Msun
    R_star = 1.0 * Rsun
    T_eff = 4000
    L_star = 4 * np.pi * R_star ** 2 * sig_SB * T_eff ** 4 / Lsun
    return 4.871e-7 * (tau_s(R, Sigma_G, T) / 0.01) * M_star ** (-17/7) * L_star ** (6/7) * R ** (12/7)

def tau_s(R, Sigma_G, T):
    """Calculates the Stokes number (Lorek et al 2022)."""
    return np.min((tau_s_frag(T), tau_s_drift(R, Sigma_G, T)), axis = 0)

def tau_s_frag(T):
    """Calculates the fragmentation limited Stokes number (Lorek et al 2022)."""
    v_frag = 500
    return v_frag ** 2 / (2 * alpha * c_s(T) ** 2)

def tau_s_drift(R, Sigma_G, T):
    """Calculates the drift limited Stokes number (Lorek et al 2022)."""
    epsilon_g = 0.5
    d2g = 0.01
    return 3 * np.sqrt(np.pi) / 4 * epsilon_g * d2g / eta(R, Sigma_G, T)

def eta(R, Sigma_G, T):
    """Calculates the eta parameter (Fortier et al 2012)."""
    return -1 * dP_dR(R, Sigma_G, T) / (2 * Omega_K(R) ** 2 * R * AU * rho_mid(R, Sigma_G, T))

## Pebble accretion transition mass

def M_tr(R):
    """Calculates the transition mass from the Bondi to the Hill regime in Earth masses (Lorek et al 2022)."""
    M_star = 1.0  # in units of Msun
    R_star = 1.0 * Rsun
    T_eff = 4000
    L_star = 4 * np.pi * R_star ** 2 * sig_SB * T_eff ** 4 / Lsun
    return 1.125e-4 * M_star ** (-17/7) * L_star ** (6/7) * R ** (12/7)

## Pebble isolation mass

def M_iso_peb(R, Sigma_G, T):
    """Calculates the pebble isolation mass in Earth masses (Lorek et al 2022)."""
    return 25 * (H(R, T) / R / 0.05) ** 3 * (0.34 * (-3 / np.log10(alpha)) ** 4 + 0.66 ) * (1 - (dlnP_dlnR(R, Sigma_G, T) + 2.5) / 6)

## Planetesimal isolation mass

def M_iso_pltsml(R, Sigma_plantesimal):
    """Calculates the planetesimal isolation mass in Earth masses (Rafikov 2011)."""
    b_tilde = 10
    M_star = 1.0 * Msun
    return (2 * np.pi * (R * AU) ** 2 * b_tilde * Sigma_plantesimal) ** (3/2) * (3 * M_star) ** (-1/2) / Mearth

In [61]:
## Animate pebble accretion regimes (one frame per timestep)

n_times = len(t_interval)

y_min, y_max = 1e-12, 1e4
M_tr_vals = M_tr(R)

fig, ax = plt.subplots(figsize = (8, 6))

def update(i):
    ax.clear()

    M_birth_vals = M_birth(R, Sigma_G[i], T[i])
    M_on_vals = M_on(R, Sigma_G[i], T[i])
    M_iso_peb_vals = M_iso_peb(R, Sigma_G[i], T[i])
    M_iso_pltsml_vals = M_iso_pltsml(R, Sigma_planetesimal[i])

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_ylim(y_min, y_max)

    b0 = np.full_like(R, y_min)
    b1 = np.maximum(b0, M_on_vals)
    b2 = np.maximum(b1, M_tr_vals)
    b3 = np.maximum(b2, M_iso_peb_vals)
    b4 = np.full_like(R, y_max)

    ax.fill_between(R, b0, b1, color = 'red', alpha = 0.5)
    ax.fill_between(R, b1, b2, color = 'yellow', alpha = 0.5)
    ax.fill_between(R, b2, b3, color = 'green', alpha = 0.5)
    ax.fill_between(R, b3, b4, color = 'red', alpha = 0.5)

    ax.plot(R, M_birth_vals, color = 'black', label = r'$M_{birth}$')
    ax.plot(R, M_iso_pltsml_vals, color = 'black', linestyle = '--', label = r'$M_{iso, pltsml}$')

    ax.text(3e-1, 1e-10, r'$M < M_{on}$', ha = 'center', va = 'center', bbox = dict(boxstyle = 'round', facecolor = 'white', alpha = 0.8, edgecolor = '0.8'))
    ax.text(3e-1, 1e-6, r'$M < M_{tr}$', ha = 'center', va = 'center', bbox = dict(boxstyle = 'round', facecolor = 'white', alpha = 0.8, edgecolor = '0.8'))
    ax.text(3e-1, 3e2, r'$M > M_{iso}$', ha = 'center', va = 'center', bbox = dict(boxstyle = 'round', facecolor = 'white', alpha = 0.8, edgecolor = '0.8'))

    ax.margins(0)
    ax.set_xlabel(r'$R$ [AU]')
    ax.set_ylabel(r'$M$ [$M_\oplus$]')
    ax.set_title(f'Pebble Accretion Regimes: t = {t_interval[i]} Myr')
    ax.legend(loc = 'lower center')

update(0)
plt.tight_layout()

ani = FuncAnimation(fig, update, frames = n_times, interval = 1000, blit = False)

if save_figures:
    gif_path = os.path.join(figure_dir, "pebble_accretion_regimes.gif")
    ani.save(gif_path, writer = PillowWriter(fps = 5))

    mp4_path = os.path.join(figure_dir, "pebble_accretion_regimes.mp4")
    ani.save(mp4_path, writer = FFMpegWriter(fps = 5))

plt.close(fig)

HTML(ani.to_jshtml())